In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Functions

In [ ]:
def map_ltv_range_to_lgd_bin(flt_ltv, dict_bins_ltv):
    for flt_threshold, flt_val in dict_bins_ltv.items():
        if flt_ltv <= flt_threshold:
            return flt_val
    # else
    return np.max(list(dict_bins_ltv.values()))

In [ ]:
def get_lgd_bk_nobk(int_bk, flt_lgd_bk, flt_lgd_nobk):
    # if bk
    if int_bk == 1:
        return flt_lgd_bk
    else:
        return flt_lgd_nobk

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/04_join_targets/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

#### Get Gen 13 Predictions

In [ ]:
# import parser
str_filename = 'cls_parser.pkl'
str_local_path = f'./{str_filename}'
cls_parser = pickle.load(open(str_local_path, 'rb'))

# preprocess
cls_model_preprocessing = cls_parser.cls_model_preprocessing
df = cls_model_preprocessing.transform(df)
# show
df

In [ ]:
# predict - PD
cls_model_inference = cls_parser.cls_model_inference
# get intercept
flt_intercept = cls_model_inference.intercept_[0]
# get cols in model
list_cols_model = list(cls_model_inference.feature_names_in_)
# get the coef
list_coef = list(cls_model_inference.coef_[0])
# make dict
dict_coef = dict(zip(list_cols_model, list_coef))
# get contribution
list_str_contribution = []
for key, val in dict_coef.items():
    str_contribution = f'{key}_contribution'
    df[str_contribution] = df[key] * val 
    list_str_contribution.append(str_contribution)
# get the sum
df['sum'] = df[list_str_contribution].apply(sum, axis=1)
# get the log odds
df['log_odds'] = df['sum'] + flt_intercept
# get the pd
df['gen13_pd'] = np.exp(df['log_odds']) / (1 + np.exp(df['log_odds']))

In [ ]:
# predict - LGD (BK)
dict_bins_ltv_bk = cls_parser.dict_bins_ltv_bk
df['LGD_bk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv_bk,
    ),
)

In [ ]:
dict_bins_ltv_nobk = cls_parser.dict_bins_ltv_nobk
df['LGD_nobk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv_nobk,
    ),
)

In [ ]:
# get lgd
df['gen13_lgd'] = df.apply(
    lambda x: get_lgd_bk_nobk(
        int_bk=x['ENG-bk'],
        flt_lgd_bk=x['LGD_bk'],
        flt_lgd_nobk=x['LGD_nobk'],
    ),
    axis=1,
)

#### Save to s3

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)